# SVD Explainer - v1.0

In [ ]:
import sys

!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install catboost
!{sys.executable} -m pip install nbimporter
!{sys.executable} -m pip install tensorflow

#!git clone https://github.com/AI4LIFE-GROUP/OpenXAI.git
!{sys.executable} -m pip install -e OpenXAI

In [ ]:
import time
import numpy as np
import pandas as pd
import zipfile as zf
import nbimporter

import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.sparse.linalg import eigs

import xgboost as xgb
from catboost import CatBoostClassifier

import lime
import lime.lime_tabular

import shap
shap.initjs()

# Load and process dataset

In [ ]:
import synthetic_data as syn

synth= syn.synthetic_data_generator(n=500,df=True)

In [ ]:
synth= pd.read_csv('datasets/SVD_Exp_synth_data.csv')

# split synth into features (x) and target (y)
x_syn= synth.loc[:,synth.columns[0:4]]
y_syn= synth.loc[:,synth.columns[4:5]]

x_syn.head()

In [ ]:
print(np.round(synth.memory_usage().sum() / 10**6, 2), "MB")

In [ ]:
# Normalization
from sklearn.preprocessing import MinMaxScaler

# All dataset is numeric
#scaler= MinMaxScaler(feature_range=(0, 1))
#norm_x_syn= scaler.fit_transform(np.asarray(x_syn))
#norm_x_syn= pd.DataFrame(norm_x_syn, columns=x_syn.columns)

norm_x_syn= normalize_selected(x_syn)

norm_x_syn.head()

In [ ]:
# Standarization
from sklearn.preprocessing import StandardScaler

scaler= StandardScaler().fit(np.asarray(x_syn))
# realiza a padronização (média= 0, variância= 1)
stand_x_syn= scaler.transform(np.asarray(x_syn))
stand_x_syn= pd.DataFrame(stand_x_syn, columns=x_syn.columns)

stand_x_syn.head()

In [ ]:
train, test, labels_train, labels_test= sklearn.model_selection.train_test_split(norm_x_syn,y_syn,
                                                                                 train_size=0.80,
                                                                                 random_state=1234)

# SVD Explainer Methods